In [35]:
import pandas as pd
import numpy as np
import matplotlib as plt
from razdel import tokenize
import pymorphy3
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer

In [25]:
df = pd.read_csv("../datasets/porn_detection/train.csv")
df = df.dropna(subset=['title'])

In [3]:
df.head()

,ID,url,title,label
0,0,m.kp.md,"Экс-министр экономики Молдовы - главе МИДЭИ, ц...",0
1,1,www.kp.by,Эта песня стала известна многим телезрителям б...,0
2,2,fanserials.tv,Банши 4 сезон 2 серия Бремя красоты смотреть о...,0
3,3,colorbox.spb.ru,Не Беси Меня Картинки,0
4,4,tula-sport.ru,В Новомосковске сыграют следж-хоккеисты алекси...,0


In [4]:
df[df["label"] == 1].head(10)

,ID,url,title,label
8,8,xlecx.com,league of legends » Page 5 » Porn comics free ...,1
12,12,pornmult.info,"кримпай,мать и сын » Страница 5 » смотреть пор...",1
19,19,24eropixel.net,Мужик поставил блондинку раком и отодрал ее ту...,1
21,21,gdespaces.com,Порно которое ты искал / Видео - Spaces.ru / S...,1
41,41,hdxclub.com,Лесбийский секс с кунилингусом двух стройных с...,1
47,47,jrfzdohkntmopulam5635ayigseqr47ghplfa5l67uo72g...,Lesbians Monique Alexander and Violet Starr fu...,1
52,52,desixxxtube.pro,Indian aunty bjowjob and fucking with her part...,1
53,53,ipad.perfektdamen.co,"Sweet, Russian honey, Angelika got down and di...",1
55,55,daftsex.com,Playlist Lesbian — DaftSex,1
56,56,topdevka.com,Фото голых девочек и бесплатная эротика на Top...,1


In [11]:
df[df["url"].str.contains("pornuha")].sum(axis=0)

ID                                                 1199113
url      porno-pornuha.compornuha.tvporno-pornuha.compo...
title    Горячее порно с сексуальной блондинкой Alex Gr...
label                                                   15
dtype: object

In [20]:
df[(df["label"] == 0) & (df["url"].str.contains("porn"))].shape

(0, 4)

#### Заметим, что некоторые комбинации букв содержатся только в сайтах 18+ (porn, porevo)
В будущем будем все сайты с таким называнием отмечать как сайты 18+ (повысим recall)

In [26]:
# preprocessing
def tokenize_df(df):

    def lemmatize_text(text):
        tokens = [token.text for token in tokenize(text)]
        lemmas = []
        for token in tokens:
            if token.isalpha():
                parsed = morph.parse(token)[0]  # берем первый вариант разбора
                lemmas.append(parsed.normal_form)
        return " ".join(lemmas)
    
    df['tokens'] = df['title'].apply(lambda x: " ".join([token.text for token in tokenize(x)])) 
    morph = pymorphy3.MorphAnalyzer()   # эта штука может убирать названия фильмов и тд
    df['lemmatized'] = df['tokens'].apply(lemmatize_text)

In [27]:
tokenize_df(df)

In [40]:
encoder = TfidfVectorizer(max_df=0.8,
                          min_df=2,
                          max_features=1000)
tf_idf_matr = encoder.fit_transform(df["lemmatized"])

In [41]:
tf_idf_matr

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 560500 stored elements and shape (135308, 1000)>